In [1]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
warnings.filterwarnings('ignore')

dataset_number = 10
split_ratios = [0.7, 0.8]
use_tuned = True
top_mode = 'global'
res_dir = '../Results/'

base_names = ['Random Forest', 'Logistic Regression', 'Decision Tree', 'XGBoost', 'CatBoost', 'AdaBoost', 'LightGBM', 'Support Vector Machine', 'K-Nearest Neighbors', 'Extra Trees', 'Gradient Boosting', 'Naive Bayes']

best_params = json.load(open(res_dir + 'best_params.json')) if os.path.exists(res_dir + 'best_params.json') else {}
top_info = json.load(open(res_dir + 'top_classifiers.json')) if os.path.exists(res_dir + 'top_classifiers.json') else {}

In [2]:
def make_model(name, params={}):
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=3000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
        'CatBoost': CatBoostClassifier(verbose=0, allow_writing_files=False, random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'LightGBM': LGBMClassifier(verbose=-1, random_state=42),
        'Support Vector Machine': SVC(probability=True, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(),
        'Extra Trees': ExtraTreesClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Naive Bayes': GaussianNB()
    }
    model = models[name]
    if use_tuned and len(params) > 0:
        model.set_params(**params)
    return model

In [3]:
def load_split(ratio):
    folder = '../Datasets/Split_' + str(round(ratio * 100)) + '/'
    train = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_train.csv')
    test = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_test.csv')
    return train.drop('label', axis=1), test.drop('label', axis=1), train['label'], test['label'], str(ratio)

In [4]:
accuracy_results = []
confusion_matrix_results = []
auc_results = []
training_testing_times = []

def record(ratio, name, y_test, pred, score, training_time, testing_time):
    accuracy_results.append({'Split Ratio': ratio, 'Classifier': name, 'Accuracy': accuracy_score(y_test, pred)})
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    confusion_matrix_results.append({
        'Split Ratio': ratio, 'Classifier': name, 'True Positive': tp, 'False Negative': fn, 'True Negative': tn, 'False Positive': fp,
        'True Positive Rate': tp / (tp + fn), 'False Negative Rate': fn / (tp + fn), 'True Negative Rate': tn / (tn + fp), 'False Positive Rate': fp / (tn + fp)})
    auc_results.append({'Split Ratio': ratio, 'Classifier': name, 'AUC Value': roc_auc_score(y_test, score)})
    training_testing_times.append({'Split Ratio': ratio, 'Classifier': name, 'Training Time (s)': training_time, 'Testing Time (s)': testing_time})

for ratio in split_ratios:
    X_train, X_test, y_train, y_test, key = load_split(ratio)
    tuned_params = best_params.get(key, {}).get('Dataset_' + str(dataset_number), {})
    fitted = {}
    probas = {}
    preds = {}
    train_times = {}

    for clf_name in base_names:
        clf = make_model(clf_name, tuned_params.get(clf_name, {}))
        start_time = time.time()
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = clf.predict(X_test)
        proba = clf.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        fitted[clf_name] = clf
        probas[clf_name] = proba
        preds[clf_name] = pred
        train_times[clf_name] = training_time
        record(ratio, clf_name, y_test, pred, proba, training_time, testing_time)

    info = top_info.get(key, {}).get('global' if top_mode == 'global' else 'Dataset_' + str(dataset_number), {})
    for top in [3, 5, 7]:
        selected = info.get(str(top), base_names[:top])
        scores = info.get('scores', {})
        weights = np.array([scores.get(n, 1.0) for n in selected])

        stack = StackingClassifier(estimators=[(n, make_model(n, tuned_params.get(n, {}))) for n in selected], final_estimator=LogisticRegression(max_iter=3000), cv=5, n_jobs=-1)
        start_time = time.time()
        stack.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = stack.predict(X_test)
        proba = stack.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        record(ratio, 'Stacking Top ' + str(top), y_test, pred, proba, training_time, testing_time)

        start_time = time.time()
        prob_matrix = np.column_stack([probas[n] for n in selected])
        pred_matrix = np.column_stack([preds[n] for n in selected])
        votes = {
            'Hard Voting': pred_matrix.mean(axis=1),
            'Soft Voting': prob_matrix.mean(axis=1),
            'Weighted Voting': np.average(prob_matrix, axis=1, weights=weights)
        }
        vote_time = time.time() - start_time
        base_train = sum(train_times[n] for n in selected)
        for vote_name, score in votes.items():
            record(ratio, vote_name + ' Top ' + str(top), y_test, (score >= 0.5).astype(int), score, base_train, vote_time)

accuracy_df = pd.DataFrame(accuracy_results)
confusion_matrix_df = pd.DataFrame(confusion_matrix_results)
auc_df = pd.DataFrame(auc_results)
training_testing_times_df = pd.DataFrame(training_testing_times)

In [5]:
print("Accuracy:")
accuracy_df

Accuracy:


,Split Ratio,Classifier,Accuracy
0,0.7,Random Forest,0.669903
1,0.7,Logistic Regression,0.728155
2,0.7,Decision Tree,0.592233
3,0.7,XGBoost,0.689320
4,0.7,CatBoost,0.650485
5,0.7,AdaBoost,0.718447
6,0.7,LightGBM,0.699029
7,0.7,Support Vector Machine,0.679612
8,0.7,K-Nearest Neighbors,0.611650
9,0.7,Extra Trees,0.621359


In [6]:
print("\nConfusion Matrix Metrics:")
confusion_matrix_df


Confusion Matrix Metrics:


,Split Ratio,Classifier,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate
0,0.7,Random Forest,24,14,45,20,0.631579,0.368421,0.692308,0.307692
1,0.7,Logistic Regression,20,18,55,10,0.526316,0.473684,0.846154,0.153846
2,0.7,Decision Tree,18,20,43,22,0.473684,0.526316,0.661538,0.338462
3,0.7,XGBoost,24,14,47,18,0.631579,0.368421,0.723077,0.276923
4,0.7,CatBoost,22,16,45,20,0.578947,0.421053,0.692308,0.307692
5,0.7,AdaBoost,25,13,49,16,0.657895,0.342105,0.753846,0.246154
6,0.7,LightGBM,23,15,49,16,0.605263,0.394737,0.753846,0.246154
7,0.7,Support Vector Machine,14,24,56,9,0.368421,0.631579,0.861538,0.138462
8,0.7,K-Nearest Neighbors,10,28,53,12,0.263158,0.736842,0.815385,0.184615
9,0.7,Extra Trees,21,17,43,22,0.552632,0.447368,0.661538,0.338462


In [7]:
print("\nAUC Values:")
auc_df


AUC Values:


,Split Ratio,Classifier,AUC Value
0,0.7,Random Forest,0.757490
1,0.7,Logistic Regression,0.717004
2,0.7,Decision Tree,0.555466
3,0.7,XGBoost,0.709312
4,0.7,CatBoost,0.723482
5,0.7,AdaBoost,0.771862
6,0.7,LightGBM,0.746559
7,0.7,Support Vector Machine,0.712146
8,0.7,K-Nearest Neighbors,0.666599
9,0.7,Extra Trees,0.691903


In [ ]:
print("\nTraining Times:")
training_testing_times_df

In [8]:
os.makedirs(res_dir, exist_ok=True)
merged = accuracy_df.merge(auc_df, on=['Split Ratio', 'Classifier']).merge(confusion_matrix_df, on=['Split Ratio', 'Classifier']).merge(training_testing_times_df, on=['Split Ratio', 'Classifier'])
merged.to_csv(res_dir + 'Split_Dataset_' + str(dataset_number) + '.csv', index=False)
merged.sort_values('AUC Value', ascending=False).head(10)

,Split Ratio,Classifier,Accuracy,AUC Value,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate,Training Time (s),Testing Time (s)
34,0.8,Gradient Boosting,0.681159,0.784436,19,7,28,15,0.730769,0.269231,0.651163,0.348837,0.122200,0.008218
27,0.8,XGBoost,0.681159,0.777281,20,6,27,16,0.769231,0.230769,0.627907,0.372093,0.484747,0.007772
24,0.8,Random Forest,0.695652,0.775492,18,8,30,13,0.692308,0.307692,0.697674,0.302326,0.505892,0.049333
5,0.7,AdaBoost,0.718447,0.771862,25,13,49,16,0.657895,0.342105,0.753846,0.246154,0.198496,0.024141
39,0.8,Weighted Voting Top 3,0.652174,0.767442,18,8,27,16,0.692308,0.307692,0.627907,0.372093,0.978909,0.000000
38,0.8,Soft Voting Top 3,0.652174,0.767442,18,8,27,16,0.692308,0.307692,0.627907,0.372093,0.978909,0.000000
47,0.8,Weighted Voting Top 7,0.637681,0.762075,18,8,26,17,0.692308,0.307692,0.604651,0.395349,2.107094,0.000000
29,0.8,AdaBoost,0.666667,0.762075,17,9,29,14,0.653846,0.346154,0.674419,0.325581,0.169516,0.024246
46,0.8,Soft Voting Top 7,0.637681,0.761181,18,8,26,17,0.692308,0.307692,0.604651,0.395349,2.107094,0.000000
36,0.8,Stacking Top 3,0.710145,0.759392,18,8,31,12,0.692308,0.307692,0.720930,0.279070,2.544031,0.069636
